# 06 Parameter Sweeps and Interpretation

This notebook asks which assumptions make gaze matter. It runs compact sweeps over RF size, retinotopic jitter, SNR, and mapping error, then summarizes how grating `|ΔPO|` and natural-image response instability change together.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.utils import load_config
from src.simulation import run_sweep
from src.plotting import set_plot_style, plot_summary_bars

set_plot_style()
fig_dir = ROOT / 'results/figures'
fig_dir.mkdir(parents=True, exist_ok=True)

configs = [
    ROOT / 'configs/default.yaml', ROOT / 'configs/broad_rf.yaml', ROOT / 'configs/narrow_rf.yaml',
    ROOT / 'configs/low_jitter.yaml', ROOT / 'configs/high_jitter.yaml',
    ROOT / 'configs/low_snr.yaml', ROOT / 'configs/high_snr.yaml',
    ROOT / 'configs/mapping_error_none.yaml', ROOT / 'configs/mapping_error_small.yaml', ROOT / 'configs/mapping_error_medium.yaml'
]
summary = run_sweep(load_config(ROOT / 'configs/default.yaml', debug=True), configs, debug=True)
display(summary[['condition', 'median_abs_delta_po_deg', 'population_response_correlation', 'rdm_similarity_to_baseline', 'median_relative_response_change']].round(3))

## Sweep Summary Panels

Bar plots are useful for comparing conditions, while the scatter plot asks whether grating sensitivity predicts natural-image sensitivity.

In [ ]:
plot_summary_bars(summary, fig_dir / 'notebook06_sweep_bars.png')
plt.figure(figsize=(7,4.5))
sns.scatterplot(data=summary, x='median_abs_delta_po_deg', y='median_relative_response_change', hue='condition', s=90)
plt.xlabel('Median |ΔPO| (deg)')
plt.ylabel('Natural-image relative change')
plt.title('Do grating and natural-image sensitivity covary?')
plt.legend(frameon=False, bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.savefig(fig_dir / 'notebook06_sweep_bridge.png', bbox_inches='tight')

## Ranked Conditions

The ranked tables make the interpretation concrete. High `|ΔPO|` does not always mean the natural-image representation is the least stable, because the stimulus classes probe different aspects of the RF model.

In [ ]:
display(summary.sort_values('median_abs_delta_po_deg', ascending=False)[['condition', 'median_abs_delta_po_deg', 'p90_abs_delta_po_deg', 'median_tuning_strength_change']].round(3))
display(summary.sort_values('population_response_correlation')[['condition', 'population_response_correlation', 'rdm_similarity_to_baseline', 'median_relative_response_change']].round(3))

## Interpretation Checklist

- Gaze should matter little for grating PO when responses are phase-invariant, phases are densely averaged, mapping is accurate, SNR is high, RFs are broad, and preferred SFs are low.
- Gaze should matter more for grating PO when RFs are narrow, preferred SFs are higher, simple-cell phase sensitivity is sampled sparsely, mapping is imperfect, or PO estimates are noisy.
- Gaze can affect natural-image responses even when grating PO is stable, because image translation changes local luminance structure over RF subfields.
- Mapping errors matter when their displacement approaches RF subfield scale.
- SNR dominates when bootstrap PO uncertainty is large even without deterministic geometry changes.

Final answer for this null model: grating `ΔPO` is mainly a phase-sensitive estimation phenomenon, while natural-image instability reflects direct translation of structured image content across RFs. The comparison is most interpretable when the phase-sanity, mapping-error, and SNR controls are checked first.